In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class colorcontext:
    favourite_colour: str="blue"
    least_favourite_colour: str="yellow"

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=colorcontext
)

In [4]:
from langchain.messages import HumanMessage

response= agent.invoke({
    "messages":[HumanMessage(content="What is my favorite color?")]},
    context= colorcontext()
)


In [5]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is my favorite color?', additional_kwargs={}, response_metadata={}, id='9b88d2b6-5059-4372-8a5b-f34860d9cc64'),
              AIMessage(content='I don’t know your favorite color unless you tell me. Want me to guess, or do a quick little quiz to figure it out?\n\nHere’s a short 4-question mini-quiz:\n1) Do you prefer cool colors (blue/green/purple), warm colors (red/orange/yellow), or neutrals (black/gray/white)?\n2) Do you like bright/saturated colors or muted/pastel tones?\n3) Do you want a color that goes with most outfits or something that stands out?\n4) Any colors you definitely avoid?\n\nTell me your answers and I’ll suggest your likely favorite. If you’d rather, I can just pick a color at random or start with blue as a common guess.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1309, 'prompt_tokens': 12, 'total_tokens': 1321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'a

## Accessing Context

In [6]:
from langchain.tools import tool
from langchain_core.runnables import RunnableConfig

@tool
def get_favourite_colour(config: RunnableConfig) -> str:
    """Get the favourite colour of the user"""
    context = config["configurable"].get("colour_context")
    return context.favourite_colour

@tool
def get_least_favourite_colour(config: RunnableConfig) -> str:
    """Get the least favourite colour of the user"""
    context = config["configurable"].get("colour_context")
    return context.least_favourite_colour

In [ ]:
agent= create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=colorcontext
)

In [8]:
my_context = colorcontext(favourite_colour="green", least_favourite_colour="red")

response = agent.invoke(
    {
        "messages": [HumanMessage(content="What is my favourite colour?")]
    },
    config={"configurable": {"colour_context": my_context}} 
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='38f3d4ef-9979-4190-9d86-a6516cab2611'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 149, 'total_tokens': 234, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqmT2Mso1a5huOAYXdEBaMP8FSU41', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec7fd-6ef7-75a3-8db5-7aab91621332-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_ZZgcgyB1mHCQ3gks5H5gOv5o', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_to

In [15]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [21]:
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.types import Command
from langchain.messages import  ToolMessage
from typing import Annotated



@tool
def update_favourite_color(favourite_colour: str, tool_call_id: Annotated[str, InjectedToolCallId]) ->Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [ToolMessage("Successfully updated", tool_call_id=tool_call_id)]
        }
    )

In [22]:
from typing import Annotated
from langgraph.prebuilt import InjectedState

@tool
def read_favourite_color(state: Annotated[dict, InjectedState]) ->str:
    """Read the favourite colour of the user from the state."""
    return state.get("favourite_colour", "No favourite color found in state")

In [26]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint

agent= create_agent(
    model="gpt-5-nano",
    tools=[update_favourite_color, read_favourite_color],
    checkpointer= InMemorySaver(),
    state_schema=CustomState
)


In [27]:
response= agent.invoke(
    {
        "messages": [HumanMessage(content="My favourite colour is green")]
    },
    {
        "configurable":{
            "thread_id":"1"
        }
    }
)
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1c61bc70-c955-49b3-9bd6-89c9c5c02fa3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 162, 'total_tokens': 445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqmfKWiaGkT2cSLWZLTR6700DfD2c', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec809-0fe4-7e20-b974-b7d98ea2f93d-0', tool_calls=[{'name': 'update_favourite_color', 'args': {'favourite_colour': 'green'}, 'id': 'call_djjPCOOFw0AzEw3svjH7Pc66', 'type': 'tool_call'}], invalid_t

In [28]:
response= agent.invoke(
    {
        "messages":[HumanMessage(content="Whats my favourite colour?.")]
    },
    {
        "configurable": {
            "thread_id":"1"
        }
    }
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1c61bc70-c955-49b3-9bd6-89c9c5c02fa3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 162, 'total_tokens': 445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqmfKWiaGkT2cSLWZLTR6700DfD2c', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec809-0fe4-7e20-b974-b7d98ea2f93d-0', tool_calls=[{'name': 'update_favourite_color', 'args': {'favourite_colour': 'green'}, 'id': 'call_djjPCOOFw0AzEw3svjH7Pc66', 'type': 'tool_call'}], invalid_t